# Budgerigar Human Speech M0：有损听觉与共享运动执行器
目标是从低维、有损的耳蜗感觉状态重新发出声学上相似且可懂的声音；不要求波形或绝对相位一致。

In [ ]:
#@title 1. 更新项目
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib,shutil,datetime,json
repo=Path(REPO_DIR)
if not (repo/'.git').is_dir():subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else:
 pull=subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],text=True,capture_output=True);print(pull.stdout,pull.stderr)
 if pull.returncode:
  backup=repo.with_name(f'Budgerigar_backup_{datetime.datetime.now():%H%M%S}');shutil.move(str(repo),str(backup));subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train]'],check=True);sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]:del sys.modules[name]
importlib.invalidate_caches();print('commit:',subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

In [ ]:
#@title 2. Drive、低维约束与流式检查
from google.colab import drive
drive.mount('/content/drive');WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar');MANIFEST=WORK_ROOT/'manifests'/'fsdd.jsonl';EVALUATOR_CHECKPOINT=WORK_ROOT/'checkpoints'/'frozen_real_audio_digit_evaluator'/'best.pt'
import torch
if not torch.cuda.is_available():raise RuntimeError('请选择 GPU runtime')
assert EVALUATOR_CHECKPOINT.is_file(),'缺少已经通过的冻结听觉评估器'
from budgerigar.human_speech_model import HumanSpeechConfig,create_human_speech_model
config=HumanSpeechConfig();model=create_human_speech_model(config).cuda().eval();dummy=torch.randn(2,12,config.tick_samples,device='cuda')
with torch.no_grad():full,_,diagnostics=model(dummy);state=None;parts=[]
with torch.no_grad():
 for index in range(dummy.shape[1]):value,state,_=model.stream_step(dummy[:,index],state);parts.append(value)
streamed=torch.stack(parts,1);maximum=float((full-streamed).abs().max());mean=float((full-streamed).abs().mean());raw_rate=config.sample_rate;latent_rate=config.latent_dim*config.subframes*config.sample_rate/config.tick_samples;print('parameters:',sum(p.numel() for p in model.parameters()),'raw/latent values per second:',raw_rate,latent_rate,'stream max/mean:',maximum,mean);assert config.latent_dim<config.tick_samples//config.subframes and maximum<1e-4

In [ ]:
#@title 3. 平衡声门激励：从原始 500-step 分支训练
MAX_STEPS=800 #@param {type:'integer'}
BATCH_SIZE=8 #@param {type:'integer'}
from budgerigar.train_human_speech import HumanSpeechTrainingConfig,train_human_speech
SOURCE_RUN=WORK_ROOT/'checkpoints'/'human_cochlear_shared_motor_m0';RUN_DIR=WORK_ROOT/'checkpoints'/'human_cochlear_balanced_larynx_m0';RESUME_FROM=SOURCE_RUN/'last.pt'
training=HumanSpeechTrainingConfig(batch_size=BATCH_SIZE,max_steps=MAX_STEPS,max_train_records=1000,max_validation_records=100,learning_rate=2e-4,control_weight=1.0)
report=train_human_speech(MANIFEST,RUN_DIR,EVALUATOR_CHECKPOINT,training,config,resume_from=RESUME_FROM);print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 4. 声学相似审计与试听（只比较本轮新历史）
assert report['history'],'本轮没有产生新的验证记录；请确认 MAX_STEPS 大于源检查点 step。'
def balance_score(x):
    import math
    return x['validation_acoustic_loss']+.5*(1-x['validation_frozen_output_digit_accuracy'])+.001*x['validation_f0_mae_hz']+.1*abs(math.log(max(x['validation_spectral_centroid_ratio'],1e-6)))+.05*abs(math.log(max(x['validation_high_frequency_energy_ratio'],1e-6)))
best=min(report['history'],key=balance_score)
m0_pass=best['validation_frozen_real_digit_accuracy']>.9 and best['validation_frozen_output_digit_accuracy']>.8 and best['validation_output_separation_ratio']>.5 and best['validation_shuffled_relative_degradation']>.1 and best['validation_mean_relative_degradation']>.1 and best['validation_f0_mae_hz']<60 and .7<best['validation_voiced_duration_ratio']<1.3 and .5<best['validation_spectral_centroid_ratio']<1.5 and .3<best['validation_high_frequency_energy_ratio']<2
print(json.dumps(best,ensure_ascii=False,indent=2));print('m0_pass =',m0_pass)
payload=torch.load(RUN_DIR/'validation_example.pt',map_location='cpu',weights_only=False);import torchaudio
INPUT=RUN_DIR/'human_input.wav';OUTPUT=RUN_DIR/'human_resynthesis.wav';torchaudio.save(str(INPUT),payload['input'].unsqueeze(0),payload['sample_rate']);torchaudio.save(str(OUTPUT),payload['reconstruction'].unsqueeze(0),payload['sample_rate'])
from IPython.display import Audio,display
print('输入：');display(Audio(filename=str(INPUT)));print('模型重新发声：');display(Audio(filename=str(OUTPUT)))
if not m0_pass:print('未通过：先看输出数字准确率，再检查 F0 误差、频谱质心、高频能量和感觉消融；不要仅因频谱损失下降而扩训。')